In [ ]:
pip install langchain_google_genai

In [ ]:
!pip install -q langchain-core requests

In [ ]:
pip install langchain_google_genai langchain-core requests

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import requests

# Initialize the LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Define the tool
@tool
def multiply(a:int,b:int)->int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

# Bind the tool to the LLM
llm_with_tools=llm.bind(tools=[multiply])

# Initial human query
query = HumanMessage('can you multiply 3 with 10')

# Invoke the LLM with the human query to get a tool call suggestion
result = llm_with_tools.invoke([query])

# Extract the tool call from the LLM's response
# Check if tool_calls exist and if the multiply tool is suggested
if result.tool_calls and result.tool_calls[0]['name'] == 'multiply':
    tool_call_details = result.tool_calls[0]
    tool_args = tool_call_details['args']
    tool_call_id = tool_call_details['id']

    # Execute the tool with the extracted arguments
    tool_result_value = multiply.invoke(tool_args)

    # Create a ToolMessage with the result of the tool execution
    tool_message = ToolMessage(content=str(tool_result_value), name='multiply', tool_call_id=tool_call_id)

    # Prepare the messages for the final LLM invocation:
    # 1. Original human query
    # 2. LLM's response (AIMessage with the tool call)
    # 3. Tool's output (ToolMessage)
    final_messages = [
        query,
        result, # This is the AIMessage containing the tool_calls
        tool_message
    ]

    # Invoke the LLM again with the complete conversation history
    final_response = llm_with_tools.invoke(final_messages)
    print(final_response.content)
else:
    print(result.content)
